# PCA analysis pipeline

This notebook performs the core PCA workflow on cleaned driving behavior data.

**Input:**

- `data/EEDA_cleaned.csv` (generated by `0_preprocessing.ipynb`)

**Main outputs:**

Saved under `results/` (organized by event via helper functions):
- cleaned subsets for PCA
- variances and eigenvalues
- PCA components/loadings
- biplots and cos2 plots

**Run context**

Run this notebook from the project root directory so relative paths resolve correctly.

In [1]:
import gc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from xarray.util.generate_ops import inplace

warnings.simplefilter(action='ignore', category=FutureWarning)
import dataframe_image as dfi
import os
import math
from  matplotlib.ticker import PercentFormatter
sns.set_style("white")
# sns.set_context("notebook")
sns.set_theme(style="ticks", palette="husl")
from pathlib import Path
import cv2
# set dask dashboard
import dask
import dask.array as da
dask.config.set({'dataframe.query-planning': True})
import dask.dataframe as dd
from scipy.spatial.transform import Rotation as R
# pd.options.mode.chained_assignment = None  # default='warn'
from dask.distributed import Client
client = Client()  # start distributed scheduler locally.
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 64.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:53550,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 16
Started: Just now,Total memory: 64.00 GiB
Comm: tcp://127.0.0.1:53562,Total threads: 4
Dashboard: http://127.0.0.1:53567/status,Memory: 16.00 GiB
Nanny: tcp://127.0.0.1:53553,


# 1. Load data and resampling

In [2]:
# Check and create 'data' and 'results' folders if they do not exist
data_folder = 'data'
results_folder = 'results'

if not os.path.exists(data_folder):
    os.makedirs(data_folder)
if not os.path.exists(results_folder):
    os.makedirs(results_folder)

# Assign folder paths to variables for later use
DATA_DIR = data_folder
RESULTS_DIR = results_folder

# Load cleaned data 
# Columns to load
cols = ['timestamp_REF','UnixTimeStamp','TobiiTimeStamp','Auto_TobiiTimeStamp',
        'uid', 'Condition','Event','eye_theta_h','eye_theta_v','streeringDegree',
        'CarYaw_degrees','CarRoll_degrees', 'CarPitch_degrees','RelativeHeadYaw_degrees',
        'RelativeHeadPitch_degrees', 'RelativeHeadRoll_degrees']
df = dd.read_csv(os.path.join(DATA_DIR, 'EEDA_cleaned.csv'), usecols=cols).compute()
df.head()

,timestamp_REF,UnixTimeStamp,TobiiTimeStamp,uid,Auto_TobiiTimeStamp,Condition,Event,streeringDegree,eye_theta_h,eye_theta_v,CarYaw_degrees,CarPitch_degrees,CarRoll_degrees,RelativeHeadYaw_degrees,RelativeHeadPitch_degrees,RelativeHeadRoll_degrees
0,1970-01-01 02:56:39.665039062+00:00,1.630156e+09,10599.665039,017397ca31114170abc2ff61a217460b,10599.665039,Autonomous,NoEv,-42.106627,-1.611504,-3.784578,74.761305,0.847228,0.280716,3.099670,0.210684,5.025472
1,1970-01-01 02:56:39.686523437+00:00,1.630156e+09,10599.686523,017397ca31114170abc2ff61a217460b,10599.686523,Autonomous,NoEv,-42.106627,-1.539683,-3.805523,74.720284,0.850861,0.273172,2.883921,-0.249197,4.662842
2,1970-01-01 02:56:39.699218750+00:00,1.630156e+09,10599.686523,017397ca31114170abc2ff61a217460b,10599.699219,Autonomous,NoEv,-42.106627,-1.539683,-3.805523,74.720284,0.850861,0.273172,2.883921,-0.249197,4.662842
3,1970-01-01 02:56:39.721679687+00:00,1.630156e+09,10599.721680,017397ca31114170abc2ff61a217460b,10599.721680,Autonomous,NoEv,-42.132882,-1.098943,-3.900335,74.650599,0.830270,0.260069,2.911437,-0.207017,4.638311
4,1970-01-01 02:56:39.743164062+00:00,1.630156e+09,10599.743164,017397ca31114170abc2ff61a217460b,10599.743164,Autonomous,NoEv,-42.132882,-0.994679,-3.919528,74.621516,0.819260,0.252394,2.890011,-0.181404,4.623228


Inspect original sampling rate

In [3]:
# Ensure datetime for timestamp reference
df['timestamp_REF'] = pd.to_datetime(df.Auto_TobiiTimeStamp, utc=True, unit='s', origin='unix')

In [4]:
# Function to calculate time passed in seconds from the first timestamp
def calculate_time_passed(group):
    first_timestamp = group['timestamp_REF'].min()
    group['time'] = (group['timestamp_REF'] - first_timestamp).dt.total_seconds().round(2)
    return group

df = df.groupby('uid').apply(calculate_time_passed).reset_index(drop=True)
df.head()

,timestamp_REF,UnixTimeStamp,TobiiTimeStamp,uid,Auto_TobiiTimeStamp,Condition,Event,streeringDegree,eye_theta_h,eye_theta_v,CarYaw_degrees,CarPitch_degrees,CarRoll_degrees,RelativeHeadYaw_degrees,RelativeHeadPitch_degrees,RelativeHeadRoll_degrees,time
0,1970-01-01 02:56:39.665039062+00:00,1.630156e+09,10599.665039,017397ca31114170abc2ff61a217460b,10599.665039,Autonomous,NoEv,-42.106627,-1.611504,-3.784578,74.761305,0.847228,0.280716,3.099670,0.210684,5.025472,0.00
1,1970-01-01 02:56:39.686523437+00:00,1.630156e+09,10599.686523,017397ca31114170abc2ff61a217460b,10599.686523,Autonomous,NoEv,-42.106627,-1.539683,-3.805523,74.720284,0.850861,0.273172,2.883921,-0.249197,4.662842,0.02
2,1970-01-01 02:56:39.699218750+00:00,1.630156e+09,10599.686523,017397ca31114170abc2ff61a217460b,10599.699219,Autonomous,NoEv,-42.106627,-1.539683,-3.805523,74.720284,0.850861,0.273172,2.883921,-0.249197,4.662842,0.03
3,1970-01-01 02:56:39.721679687+00:00,1.630156e+09,10599.721680,017397ca31114170abc2ff61a217460b,10599.721680,Autonomous,NoEv,-42.132882,-1.098943,-3.900335,74.650599,0.830270,0.260069,2.911437,-0.207017,4.638311,0.06
4,1970-01-01 02:56:39.743164062+00:00,1.630156e+09,10599.743164,017397ca31114170abc2ff61a217460b,10599.743164,Autonomous,NoEv,-42.132882,-0.994679,-3.919528,74.621516,0.819260,0.252394,2.890011,-0.181404,4.623228,0.08


In [5]:
def plot_sampling_rate_and_time_diff(df, title='Original', uid_col='uid', timestamp_col='timestamp_REF', time_col='time'):
    """
    Plot sampling rate and time difference over time for a given dataframe.
    Args:
        df (pd.DataFrame): The dataframe to plot.
        title (str): The title of the plot.
        uid_col (str): The column name of the uid.
        timestamp_col (str): The column name of the timestamp.
        time_col (str): The column name of the time.
    """
    data = df.copy()
    data[timestamp_col] = pd.to_datetime(data[timestamp_col], errors='coerce')
    data = data.sort_values([uid_col, timestamp_col])

    data['time_diffs'] = data.groupby(uid_col)[timestamp_col].diff().dt.total_seconds()
    data.loc[data['time_diffs'] <= 0, 'time_diffs'] = np.nan
    data['sampling_rate'] = 1.0 / data['time_diffs']

    # Plot 1: Sampling rate per participant
    fig1, ax1 = plt.subplots(figsize=(16, 6))
    sns.boxplot(x=uid_col, y='sampling_rate', data=data, ax=ax1)
    ax1.set_title(f'{title} Sampling Rate by UID')
    ax1.set_xlabel('UID')
    ax1.set_ylabel('Sampling Rate (samples/sec)')
    ax1.tick_params(axis='x', rotation=80)
    fig1.tight_layout()

    # Plot 2: Time difference over time
    fig2, ax2 = plt.subplots(figsize=(6, 2))
    sns.scatterplot(x=time_col, y='time_diffs', data=data, s=10, ax=ax2)
    ax2.set_title(f'{title} Time Difference Over Time')
    ax2.set_xlabel('Time from start (s)')
    ax2.set_ylabel('Δt (s)')
    fig2.tight_layout()

In [23]:
# Visualize original sampling rate and timestamp differences over time
# plot_sampling_rate_and_time_diff(df)

## Choose data subset
We keep only variables to use for analysis and resample to 50Hz

In [6]:
# Drop unnecessary columns
df = df[['timestamp_REF','uid','Event','eye_theta_h', 'eye_theta_v',
                'CarYaw_degrees', 'CarPitch_degrees','CarRoll_degrees',
                'RelativeHeadYaw_degrees', 'RelativeHeadPitch_degrees',
                'RelativeHeadRoll_degrees', 'streeringDegree','Condition']]
df.head()

,timestamp_REF,uid,Event,eye_theta_h,eye_theta_v,CarYaw_degrees,CarPitch_degrees,CarRoll_degrees,RelativeHeadYaw_degrees,RelativeHeadPitch_degrees,RelativeHeadRoll_degrees,streeringDegree,Condition
0,1970-01-01 02:56:39.665039062+00:00,017397ca31114170abc2ff61a217460b,NoEv,-1.611504,-3.784578,74.761305,0.847228,0.280716,3.099670,0.210684,5.025472,-42.106627,Autonomous
1,1970-01-01 02:56:39.686523437+00:00,017397ca31114170abc2ff61a217460b,NoEv,-1.539683,-3.805523,74.720284,0.850861,0.273172,2.883921,-0.249197,4.662842,-42.106627,Autonomous
2,1970-01-01 02:56:39.699218750+00:00,017397ca31114170abc2ff61a217460b,NoEv,-1.539683,-3.805523,74.720284,0.850861,0.273172,2.883921,-0.249197,4.662842,-42.106627,Autonomous
3,1970-01-01 02:56:39.721679687+00:00,017397ca31114170abc2ff61a217460b,NoEv,-1.098943,-3.900335,74.650599,0.830270,0.260069,2.911437,-0.207017,4.638311,-42.132882,Autonomous
4,1970-01-01 02:56:39.743164062+00:00,017397ca31114170abc2ff61a217460b,NoEv,-0.994679,-3.919528,74.621516,0.819260,0.252394,2.890011,-0.181404,4.623228,-42.132882,Autonomous


In [7]:
# Define custom aggregator functions
aggregation_dict = {
        'Event': 'first',
        'eye_theta_h': 'mean',
        'eye_theta_v': 'mean',
        'CarYaw_degrees': 'mean',
        'CarPitch_degrees': 'mean',
        'CarRoll_degrees': 'mean',
        'RelativeHeadYaw_degrees':'mean',
        'RelativeHeadPitch_degrees':'mean',
        'RelativeHeadRoll_degrees':'mean',
        'streeringDegree':'mean',
        'Condition':'first',
    }

In [8]:
df.columns

Index(['timestamp_REF', 'uid', 'Event', 'eye_theta_h', 'eye_theta_v',
       'CarYaw_degrees', 'CarPitch_degrees', 'CarRoll_degrees',
       'RelativeHeadYaw_degrees', 'RelativeHeadPitch_degrees',
       'RelativeHeadRoll_degrees', 'streeringDegree', 'Condition'],
      dtype='object')

## Resampling
Run the commented code in case you suspect of repeated samples (two samples with the same timestamp)

In [82]:
# Define the columns you want to process
# columns_to_process = ['eye_theta_h', 'eye_theta_v', 'CarYaw_degrees',
#                       'CarPitch_degrees', 'CarRoll_degrees',
#                       'RelativeHeadYaw_degrees', 'RelativeHeadPitch_degrees',
#                       'RelativeHeadRoll_degrees']

# # Function to handle repeated samples on a column
# df = df.groupby('uid').apply(
#     lambda group: group.assign(
#         **{col: group[col].where(~group[col].duplicated(keep='first'), np.nan) for col in columns_to_process}
#     )
# ).reset_index(drop=True)

# # Interpolate NaN values linearly for the specified columns
# df[columns_to_process] = df[columns_to_process].interpolate(method='linear')

# # Display the cleaned DataFrame
# df.head()

In [9]:
# Define the resampling function
def resample_data_by_uid(df):
    """
    Resamples data from high frequency to 50Hz for each UID.
    Numerical columns are aggregated, and non-numerical are transformed.
    """
    # Group by 'uid' and apply the resampling
    resampled_groups = df.groupby('uid').apply(
        # Convert columns of object dtype to their inferred types
        lambda group: group.set_index('timestamp_REF').resample('20L').agg(aggregation_dict).infer_objects(copy=False).interpolate(method='linear').ffill().bfill()
    ).reset_index()

    return resampled_groups

# Only delete df after resampling, and check if df exists before calling the function
if 'df' in locals():
    # Resample the data to 50 Hz for each UID
    resampled_df = resample_data_by_uid(df)
    # Drop the original DataFrame to free up memory
    del df
    print("Your df was deleted to free memory, now you are working with 'resampled_df'.")
else:
    print("df is not defined. Please make sure your original DataFrame is loaded before resampling.")

Your df was deleted to free memory, now you are working with 'resampled_df'.


In [10]:
# Calculate time passed in seconds from the first timestamp
resampled_df = resampled_df.groupby('uid').apply(calculate_time_passed).reset_index(drop=True)
resampled_df.head()

,uid,timestamp_REF,Event,eye_theta_h,eye_theta_v,CarYaw_degrees,CarPitch_degrees,CarRoll_degrees,RelativeHeadYaw_degrees,RelativeHeadPitch_degrees,RelativeHeadRoll_degrees,streeringDegree,Condition,time
0,017397ca31114170abc2ff61a217460b,1970-01-01 02:56:39.660000+00:00,NoEv,-1.611504,-3.784578,74.761305,0.847228,0.280716,3.099670,0.210684,5.025472,-42.106627,Autonomous,0.00
1,017397ca31114170abc2ff61a217460b,1970-01-01 02:56:39.680000+00:00,NoEv,-1.539683,-3.805523,74.720284,0.850861,0.273172,2.883921,-0.249197,4.662842,-42.106627,Autonomous,0.02
2,017397ca31114170abc2ff61a217460b,1970-01-01 02:56:39.700000+00:00,NoEv,-1.319313,-3.852929,74.685442,0.840566,0.266620,2.897679,-0.228107,4.650577,-42.119755,Autonomous,0.04
3,017397ca31114170abc2ff61a217460b,1970-01-01 02:56:39.720000+00:00,NoEv,-1.098943,-3.900335,74.650599,0.830270,0.260069,2.911437,-0.207017,4.638311,-42.132882,Autonomous,0.06
4,017397ca31114170abc2ff61a217460b,1970-01-01 02:56:39.740000+00:00,NoEv,-0.994679,-3.919528,74.621516,0.819260,0.252394,2.890011,-0.181404,4.623228,-42.132882,Autonomous,0.08


In [24]:
# plot_sampling_rate_and_time_diff(resampled_df, title='Resampled')

# 2. Applying PCA
Useful packages and functions

In [25]:
import scipy as sp
from scipy.stats import chi2
from sklearn.covariance import MinCovDet
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm  # library for progress bar
import time  # simulating elapsed time 

In [26]:
# Robust Mahalonibis Distance
def robust_mahalanobis_method_dask(df, md_name='',outlier_name='', p_md_name='', cut=0.001):
    """
    Calculate a robust version of Mahalanobis distances for each data subset,
    using the Minimum Covariance Determinant (MCD) method.
    Args:
        df (pd.DataFrame): Dataframe to calculate Mahalanobis distances for.
        md_name (str): Name of the Mahalanobis distance column.
        outlier_name (str): Name of the outlier column.
        p_md_name (str): Name of the probability of the Mahalanobis distance column.
        cut (float): Cut-off point for the Mahalanobis distance.
    Returns:
        pd.DataFrame: DataFrame with Mahalanobis distances and outlier flags
    """
    #Minimum covariance determinant
    rng = np.random.RandomState(0)
    real_cov = np.cov(df.values.T)
    # print(df.columns)
    # print(real_cov)
    # Check if the covariance matrix is all zeros
    if np.all(real_cov == 0):
        # Check the dimensions of real_cov
        if real_cov.ndim == 0:  # Only one variable present
        # Create a 1x1 perturbation matrix
            perturbation = np.array(1e-2)  # Increase here
        else:  # Multiple variables present
            # Add a larger perturbation to the diagonal
            perturbation = np.eye(real_cov.shape[0]) * 1e-2  # Increase here
        real_cov += perturbation
        # print(f'PERTURBATION!: {real_cov}')
    # Calculate inverse covariance
    if df.shape[1] < 2:
        # Reshape when only one column in used (e.g., 'SteeringInput')
        X = rng.multivariate_normal(mean=np.mean(df, axis=0), cov=real_cov.reshape((1,1)), 
            size= round(len(df) * 0.5)) # 50% of the data
        cov = MinCovDet(random_state=0,support_fraction=0.8).fit(X) #calculate covariance
        mcd = cov.covariance_ #robust covariance metric
        robust_mean = cov.location_  #robust mean
        inv_covmat = sp.linalg.inv(mcd) #inverse of covariance matrix
    else:
        X = rng.multivariate_normal(mean=np.mean(df, axis=0), cov=real_cov, size= round(len(df) * 0.5))
        cov = MinCovDet(random_state=0,support_fraction=0.8).fit(X) #calculate covariance
        mcd = cov.covariance_ #robust covariance metric
        robust_mean = cov.location_  #robust mean
        inv_covmat = sp.linalg.inv(mcd) #inverse of covariance matrix

    # Robust M-Distance
    x_minus_mu = df - robust_mean
    # Transform data into dask arrays
    x_minus_mu_dask = da.from_array(x_minus_mu.to_numpy(), chunks=(min(x_minus_mu.shape[0], 10000), df.shape[1]))
    mahal = da.sqrt(da.diagonal(da.dot(da.dot(x_minus_mu_dask,inv_covmat), x_minus_mu_dask.T)))
    # Calculate md
    md = mahal.compute()
    # Compute the chi-squared cumulative probability distribution to transfer the md2 into probabilities
    probability_md = 1 - chi2.cdf(md, df=df.shape[1])

    # Save md values and probabilities to df column
    md_df = pd.DataFrame({md_name:md,p_md_name:probability_md})
    # Set a Chi2 cut-off point using probability of 0.01 (99.5% Chi2 quantile)
    # Degrees of freedom (df) = number of variables
    threshold = chi2.ppf((1-cut), df=df.shape[1])
    # STD threshold, assuming 'md' contains your computed distances
    # mean_md = np.mean(md)
    # std_md = np.std(md)
    # threshold_3std = mean_md + 4 * std_md
    # Flag outliers as md > threshold
    md_df[outlier_name] = md_df[md_name] > threshold
    return md_df

In [27]:
# Mahalanobis distance fucntion to be used in the PCA analysis
def mahalanobis_for_pca(df):
    """
    Calculate Mahalanobis distances for each data subset.
    Args:
        df (pd.DataFrame): Input DataFrame containing the data to process
    Returns:
        pd.DataFrame: DataFrame with Mahalanobis distances and outlier flags 
        for all set of variables
    """
    ## ---- EYE columns ----
    mds_eye = robust_mahalanobis_method_dask(df=df[['eye_theta_h', 'eye_theta_v']], md_name='md_eye',
    outlier_name='eye_outlier', p_md_name='p_md_eye', cut=0.10).reset_index(drop=True)
    ## ---- HEAD columns ----
    mds_head = robust_mahalanobis_method_dask(df=df[['RelativeHeadYaw_degrees','RelativeHeadPitch_degrees',
     'RelativeHeadRoll_degrees']], md_name='md_head',outlier_name='head_outlier', p_md_name='p_md_head', cut=0.20).reset_index(drop=True)
    ## ---- CAR columns ----
    mds_car = robust_mahalanobis_method_dask(df=df[['CarYaw_degrees', 'CarPitch_degrees', 'CarRoll_degrees']],
     md_name='md_car',outlier_name='car_outlier', p_md_name='p_md_car', cut=0.10).reset_index(drop=True)
    ## ---- Steering ----
    mds_steer = robust_mahalanobis_method_dask(df=df[['streeringDegree']], md_name='md_steering',
    outlier_name='steering_outlier', p_md_name='p_md_steer', cut=0.10).reset_index(drop=True)

    ## save md and outlier data
    md_df = pd.concat([df.reset_index(drop=True),mds_eye, mds_head, mds_car,mds_steer], axis=1)
    final_df = md_df.drop(columns=['md_eye','p_md_eye','md_head','p_md_head','md_car','p_md_car',
    'md_steering','p_md_steer'])
    return final_df

# Function to interpolate NaN values in a subset of data
def interpolate_nan_pca(df):
    """
    Interpolate NaN values in a subset of data.
    Args:
        df (pd.DataFrame): Input DataFrame containing the data to process
    Returns:
        pd.DataFrame: DataFrame with interpolated NaN values
    """
    # 1. Convert 'outlier' column values to NaN where True
    # -- Eye
    df.loc[df['eye_outlier'], ['eye_theta_h', 'eye_theta_v']] = np.nan

    # -- Head
    df.loc[df['head_outlier'], ['RelativeHeadYaw_degrees', 'RelativeHeadPitch_degrees','RelativeHeadRoll_degrees']] = np.nan
    # -- Car
    df.loc[df['car_outlier'], ['CarRoll_degrees','CarYaw_degrees', 'CarPitch_degrees']] = np.nan
    # -- Steering
    df.loc[df['steering_outlier'], ['streeringDegree']] = np.nan
    # Group by 'uid' and interpolate NaN values
    interpolated_df = df.groupby('conditions').apply(lambda group: group.interpolate(method='linear').ffill().bfill()).reset_index(drop=True)
    # cleaned_df = interpolated_df.drop(columns=['eye_outlier','head_outlier','car_outlier','steering_outlier','pupil_dilation_outlier'])
    return interpolated_df

# Function to clean a subset of data by detecting and interpolating outliers during PCA analysis
def clean_subset(df):
    """
    Clean a subset of data by detecting and interpolating outliers.
    Args:
        df (pd.DataFrame): Input DataFrame containing the data to process
    Returns:
        pd.DataFrame: DataFrame with interpolated NaN values
    """
    # Detect outliers in subset by md
    subset_md = df.pipe(mahalanobis_for_pca)
    # Interpolate detected outliers
    subset_interpolated = subset_md.pipe(interpolate_nan_pca)
    return subset_interpolated


In [28]:
def init_results_dirs(base_dir: str = "results", event: str | None = None):
    """
    Initialize results directory structure with per-event subfolders.

    Structure created:
        results/
          cleaned_new/ <event_dir>
          variances/   <event_dir>
          components/  <event_dir>
          eigenvalues/ <event_dir>
          biplots/     <event_dir>
          cos_plots/   <event_dir>

    Where <event_dir> is "event_<event>" if event is provided, otherwise "all_events".

    Args:
        base_dir: Base results directory.
        event: Optional event name.

    Returns:
        dict: Mapping of category name to absolute event-specific directory path. Includes key "base".
    """
    base_abs = os.path.abspath(base_dir)
    os.makedirs(base_abs, exist_ok=True)

    event_dirname = f"event_{event}" if (event and len(str(event)) > 0) else "all_events"

    categories = [
        "cleaned_new",
        "variances",
        "components",
        "eigenvalues",
        "biplots",
        "cos_plots",
    ]

    dirs: dict[str, str] = {"base": base_abs}
    for cat in categories:
        cat_parent = os.path.join(base_abs, cat)
        os.makedirs(cat_parent, exist_ok=True)
        cat_event_dir = os.path.join(cat_parent, event_dirname)
        os.makedirs(cat_event_dir, exist_ok=True)
        dirs[cat] = cat_event_dir

    return dirs

In [29]:
# Mapping of old labels to new labels
def map_labels(labels, label_mapping):
    """
    Maps each label in the given list using the provided mapping dictionary.

    Args:
        labels (list): List of original labels.
        label_mapping (dict): Mapping dictionary for label replacement.

    Returns:
        list: New list with mapped labels.
    """
    return [label_mapping.get(label, label) for label in labels]

# Define your label mapping (you can modify this as needed)
custom_mapping = {
    'eye_theta_h': 'Eye Horizontal',
    'eye_theta_v': 'Eye Vertical',
    'RelativeHeadYaw_degrees': 'Head Yaw',
    'RelativeHeadPitch_degrees': 'Head Pitch',
    'RelativeHeadRoll_degrees': 'Head Roll',
    'CarYaw_degrees': 'Car Yaw',
    'streeringDegree': 'Steering',
    'conditions': 'Condition',
}

# Function to plot a biplot of PCA scores and loadings
def biplot(score, coef, eigenvalues,
           labels=None,
           colors=None,
           explained_variance=None,
           vector_colors=None,
           scaled=None,
           vector_linewidth=None):
    """
    Plot a biplot of PCA scores and loadings.
    Args:
        score (np.ndarray): PCA scores.
        coef (np.ndarray): PCA loadings.
        eigenvalues (np.ndarray): Eigenvalues.
        labels (list): List of original labels.
        colors (list): List of colors for each label.
        explained_variance (list): List of explained variances.
        vector_colors (list): List of colors for each vector.
        scaled (bool): Whether to scale the vectors.
        vector_linewidth (float): Linewidth for the vectors.

    Returns:
        None: Plots the biplot.

    Raises:
        TypeError: If the input is not a PCA object.
    """
    # plt.rcParams.update({'font.size': 10})
    # Apply 90-degree rotation matrix to `score` and `coef`
    # rotation_matrix = np.array([[0, -1], [1, 0]])
    # Rotate the first two components of the scores
    # score = np.dot(score[:, :2], rotation_matrix)
    # Rotate the first two components of the loadings
    # coef = np.dot(coef[:, :2], rotation_matrix)
    xs = score[:, 0]
    ys = score[:, 1]
    n = coef.shape[0]
    scalex = 1.0 / (xs.max() - xs.min())
    scaley = 1.0 / (ys.max() - ys.min())

    padding= 1.2 # 20% padding for axis and vector scaling
    padding_text = 1.1 # 10 % padding for text
    xlims = padding * np.max(np.abs(xs))
    ylims = padding * np.max(np.abs(ys))
    if colors is None:  # If no color information, plot all points gray
        plt.scatter(xs, ys, s=80, color='gray', alpha=0.5, edgecolor='gray')
    else:  # If color information is given, plot points with corresponding colors
        # plt.scatter(xs * scalex, ys * scaley, s=80, color=colors, alpha=0.5)
        plt.scatter(xs, ys, s=80, color=colors, alpha=0.5,edgecolor=colors)
    plt.xlim(-xlims * padding,xlims * padding)
    plt.ylim(-ylims* padding,ylims* padding)

    # Adjust the number of labels to match the number of coefficients
    original_labels = labels[:n]

    # Call the function to get the new mapped labels
    labels = map_labels(original_labels, custom_mapping)

    # --- Fix for arrow head aspect ratio ---
    # The problem is that plt.arrow's head_length and head_width are in data coordinates,
    # so if the axes are not square, the arrow heads get distorted.
    # Solution: Use ax.annotate with arrowprops, which handles head size in points (screen units).

    ax = plt.gca()
    # Set fixed axis limits
    fixed_x_min, fixed_x_max = -xlims, xlims  # Your specified x limits
    fixed_y_min, fixed_y_max = -ylims, ylims  # Your specified y limits

    # Set the fixed limits
    plt.xlim(fixed_x_min, fixed_x_max)
    plt.ylim(fixed_y_min, fixed_y_max)

    # Hide all spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

    # Draw principal component vectors as arrows (using annotate for consistent arrow heads)
    for i in range(n):
        # Calculate arrow end point (arrow head)
        arrow_end_x = xlims * coef[i, 0]
        arrow_end_y = ylims * coef[i, 1]
        if scaled:
            # If scaled, you may want to adjust arrow length, but here we keep as is
            pass
        # Use annotate for arrow with head in points (screen units)
        ax.annotate(
            '', xy=(arrow_end_x, arrow_end_y), xytext=(0, 0),
            arrowprops=dict(
                arrowstyle='-|>,head_width=0.15,head_length=0.2',  # consistent head size
                color=vector_colors[i],
                lw=vector_linewidth,
                alpha=1,
                shrinkA=0, shrinkB=0,
                mutation_scale=18  # controls the size of the arrow head in points
            ),
        )
        # Position the label above the arrow head
        if labels is not None:
            va = 'bottom' if arrow_end_y > 0 else 'top'
            plt.text(arrow_end_x * padding_text, arrow_end_y * padding_text,
                     f"{labels[i]}\n({coef[i, 0]:.2f}, {coef[i, 1]:.2f})", color='black', ha='center',
                     va=va, fontdict=dict(fontsize=12), bbox=dict(facecolor='white', alpha=0.0001, edgecolor='white'))

    # --- Draw axis arrows with equal length and head size using annotate (for consistent appearance) ---
    # Define the length of the axis arrows (as a fraction of the axis range)
    axis_arrow_frac = 0.25  # 25% of axis range
    axis_arrow_length_x = (fixed_x_max - fixed_x_min) * axis_arrow_frac
    axis_arrow_length_y = (fixed_y_max - fixed_y_min) * axis_arrow_frac

    # Define the starting point (bottom left corner, with a small offset)
    axis_offset_frac = 0.05  # offset from the corner (in axes-fraction units)
    start_x = fixed_x_min + axis_offset_frac * (fixed_x_max - fixed_x_min)
    start_y = fixed_y_min + axis_offset_frac * (fixed_y_max - fixed_y_min)

    # Arrow style for both axes
    axis_arrowprops = dict(
        arrowstyle='-|>,head_width=0.25,head_length=0.35',
        color='black',
        lw=2.5,
        alpha=1,
        shrinkA=0, shrinkB=0,
        mutation_scale=18  # controls the size of the arrow head in points
    )

    # Draw y-axis arrow (vertical)
    ax.annotate(
        '', xy=(start_x, start_y + axis_arrow_length_y), xytext=(start_x, start_y),
        arrowprops=axis_arrowprops
    )

    # Draw x-axis arrow (horizontal)
    x_axis_arrow_end = start_x + axis_arrow_length_x - 0.5
    ax.annotate(
        '', xy=(x_axis_arrow_end, start_y), xytext=(start_x, start_y),
        arrowprops=axis_arrowprops
    )

    # Hide tick parameters
    ax.set_xticks([])
    ax.set_yticks([])

    # Remove standard labels
    ax.set_xlabel('')
    ax.set_ylabel('')

    # Add custom positioned labels using axis-fraction coordinates for fixed placement
    label_offset_frac = 0.012  # spacing from the arrows (closer)
    # Centered under the x-axis arrow (use actual shortened arrow midpoint converted to axes fraction)
    x_axis_arrow_mid_data = (start_x + x_axis_arrow_end) / 2
    x_axis_arrow_mid_frac = (x_axis_arrow_mid_data - fixed_x_min) / (fixed_x_max - fixed_x_min)
    ax.text(
        x_axis_arrow_mid_frac,
        axis_offset_frac - label_offset_frac,
        "PC{} ({:.1f}%)".format(1, explained_variance[0] * 100),
        fontsize=12,
        ha='center', va='top', transform=ax.transAxes
    )

    # Centered to the left of the y-axis arrow
    ax.text(
        axis_offset_frac - 0.006,
        axis_arrow_frac / 2 + axis_offset_frac,
        "PC{} ({:.1f}%)".format(2, explained_variance[1] * 100),
        fontsize=12,
        ha='right', va='center', rotation=90, transform=ax.transAxes
    )

    # Add a legend for the colors if provided
    if colors is not None:
        legend_elements = [
            Line2D([0], [0],
                   marker='o',
                   color='w',
                   markerfacecolor='#2A586E',
                   markersize=12,
                   label='Manual',
                   alpha=0.8),
            Line2D([0], [0],
                   marker='o',
                   color='w',
                   markerfacecolor='#cc2936',
                   markersize=12,
                   label='Autonomous',
                   alpha=0.8)
        ]
        plt.legend(handles=legend_elements, fontsize=12)


def get_pca_var(pca, subset_scaled, feature_names):
    # Validate input types
    if not isinstance(pca, PCA):
        raise TypeError("Expected a PCA object from sklearn.decomposition.PCA")

    # Compute coordinates
    coords = pca.components_.T * np.sqrt(pca.explained_variance_)

    # Compute correlations
    # Since we're using standardized data:
    cor = coords / np.std(subset_scaled, axis=0)

    # Compute cos2 (squared loadings or cosine similarity)
    cos2 = np.square(cor)

    # Compute contributions
    total_variance = np.sum(pca.explained_variance_)
    contrib = (cos2 * 100 * pca.explained_variance_) / total_variance

    # Create dataframes with feature names as the index
    coord_df = pd.DataFrame(coords, index=feature_names, columns=[f'PC{i+1}' for i in range(pca.n_components_)])
    cor_df = pd.DataFrame(cor, index=feature_names, columns=[f'PC{i+1}' for i in range(pca.n_components_)])
    cos2_df = pd.DataFrame(cos2, index=feature_names, columns=[f'PC{i+1}' for i in range(pca.n_components_)])
    contrib_df = pd.DataFrame(contrib, index=feature_names, columns=[f'PC{i+1}' for i in range(pca.n_components_)])

    return {
        'coord': coord_df,
        'cor': cor_df,
        'cos2': cos2_df,
        'contrib': contrib_df
    }

from matplotlib.patches import Circle
from matplotlib import colors as mcolors

def plot_cosine_similarity(cos2_df, title='', event='', time_point='', save=False, save_path: str | None = None):
    """
    Plot the cosine similarity of the PCA components.
    Args:
        cos2_df (pd.DataFrame): DataFrame containing cosine similarity values.
        title (str): Title of the plot.
        event (str): Event name.
        time_point (str): Time point.
        save (bool): Whether to save the plot.

    Returns:
        None: Plots the cosine similarity plot.
    """
    # Extract feature names directly from the DataFrame index
    feature_names = cos2_df.index

    # max_cos2_value = max(cos2_df['PC1'])  # color bar maximum limit
    max_cos2_value = 1  # color bar maximum limit
    # Create a custom colormap from white to #A2530E
    cmap = mcolors.LinearSegmentedColormap.from_list("", ["white", "#A2530E"])

    features, components = cos2_df.shape
    fig, ax = plt.subplots(figsize=(10, 8))
    if len(feature_names) > 10:
        num_fontsize = 10
    num_fontsize = 14
    ax.set_title(title, fontsize=18)

    # Plot each circle and add annotations
    for i in range(features):
        for j in range(components):
            value = cos2_df.iloc[i, j]
            color = cmap(value / max_cos2_value)  # Normalize the color
            circle = Circle((j, i), radius=np.sqrt(value) * 0.5, color=color, fill=True)
            ax.add_artist(circle)

            # Annotate the cos2 value at the center of the circle
            ax.text(
                j, i, f'{value:.2f}',
                color='lightgray',
                ha='center', va='center',
                fontsize=num_fontsize,  # Adjust font size for readability
            )

    # Setup axis limits and labels
    ax.set_xlim(-0.5, components - 0.5)
    ax.set_ylim(-0.5, features - 0.5)
    ax.set_xticks(np.arange(components))
    ax.set_yticks(np.arange(features))
    ax.set_xticklabels([f'PC{i+1}' for i in range(components)], rotation=0, fontsize=16)
    ax.set_yticklabels(feature_names, fontsize=16)
    ax.set_aspect('equal', 'box')

    # Add grid lines for clarity
    ax.hlines(np.arange(-0.5, features), xmin=-0.5, xmax=components - 0.5, color='grey', lw=0.5)
    ax.vlines(np.arange(-0.5, components), ymin=-0.5, ymax=features - 0.5, color='grey', lw=0.5)

    # Set outer border to gray by modifying the spines
    for spine in ax.spines.values():
        spine.set_edgecolor('grey')
        spine.set_linewidth(0.5)

    # Add a color bar on the right
    norm = mcolors.Normalize(vmin=0, vmax=max_cos2_value)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation="vertical", fraction=0.046, pad=0.04)
    cbar.set_label('$Cos^2$', fontsize=16)

    # Increase the number of ticks on the color bar for better granularity
    num_ticks = 6
    tick_values = np.linspace(0, max_cos2_value, num_ticks)
    cbar.set_ticks(tick_values)
    cbar.set_ticklabels([round(val, 2) for val in tick_values])
    cbar.ax.tick_params(labelsize=16)  # Set colorbar tick label size larger
    cbar.outline.set_visible(True)
    plt.tight_layout()
    if save:
        # This is the default dpi for the biplots to ensure good quality videos
        # Other formats will not allow videos to be created
        # plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor(), edgecolor='none')
        plt.savefig(save_path, dpi=1000, bbox_inches='tight', facecolor=fig.get_facecolor(), edgecolor='none')
    # plt.show()

## Calculate and visualize PCA

In [17]:
# Map each condition to a float number
label_mapping = {'Manual': 1, 'Autonomous': 0}
# with outliers
resampled_df.loc[:,'conditions'] = resampled_df['Condition'].map(label_mapping)
resampled_df.groupby(['Condition'])['uid'].nunique()

Condition
Autonomous    153
Manual        131
Name: uid, dtype: int64

In [30]:
# Calculate PCA
def visualize_event_pca(df, event='', features='', scaled=True, save_results=False, interactive=True, save_format=".pdf"):
    """
    Perform PCA for each time point (or event), save all plots and results if save_results=True,
    and optionally show interactive visualization if interactive=True.

    Parameters:
        df: DataFrame
        event: str, event name to filter by
        features: list of str, features to use for PCA
        scaled: bool, whether to scale features
        save_results: bool, whether to save plots and results
        interactive: bool, whether to show interactive widgets
        save_format: str, file extension for saved images (e.g., ".pdf", ".jpeg")
    """
    # Helper to format timestamp with two digits after decimal
    def format_timestamp(ts):
        try:
            # If ts is a float or can be converted to float
            return f"{float(ts):.2f}"
        except Exception:
            # If not, fallback to string
            return str(ts)

    # Helper to create DataFrame with timestamp and event columns
    def add_time_event(df, timestamp, event_name):
        df['Time'] = timestamp
        df['Event'] = event_name
        return df

    # Helper to plot biplot (for both save and show)
    def plot_biplot(pca_result, pca, eigenvalues, original_labels, colors, explained_variance, show=False, save_path=None, timestamp=None):
        sns.set_theme(style="white")
        plt.figure(figsize=(12, 8))
        plt.title(f"Time: {format_timestamp(timestamp)}", fontsize=16)
        biplot(
            pca_result,
            pca.components_.T,
            eigenvalues,
            original_labels,
            colors,
            explained_variance,
            vector_colors=['k'] * len(original_labels),
            scaled=scaled,
            vector_linewidth=1.1
        )
        if save_path:
            # This is the default dpi for the biplots to ensure good quality videos
            # Other formats will not allow videos to be created
            plt.savefig(save_path, dpi=1200, bbox_inches='tight', edgecolor='none')

            plt.close()
        elif show:
            plt.show()

    # Helper to plot cosine similarity (for both save and show)
    def plot_cos2(pca, subset_scaled, original_labels, show=False, save_path=None, event='', timestamp=None):
        pca_cos2 = get_pca_var(pca, subset_scaled, original_labels)
        cos2_matrix = pca_cos2['cos2']
        if save_path:
            plot_cosine_similarity(
                cos2_matrix,
                title='Cosine Similarity ($Cos^2$)',
                save=True,
                event=event,
                time_point=format_timestamp(timestamp),
                save_path=save_path
            )
            plt.close()
        elif show:
            plot_cosine_similarity(
                cos2_matrix,
                title='Cosine Similarity ($Cos^2$)',
                save=False,
                event=event,
                time_point=format_timestamp(timestamp),
                save_path=None
            )
            plt.show()

    # Get timestamps for the event or all
    timestamps = df[df['Event'] == event]['time'].unique() if event else df['time'].unique()

    # DataFrames to save data
    df_cleaned_ts = pd.DataFrame()
    df_event_components = pd.DataFrame()
    df_event_variance = pd.DataFrame()
    df_pca_results = pd.DataFrame()
    df_sorted_eigenvalues = pd.DataFrame()
    df_std_dev = pd.DataFrame()

    # Cache for interactive viewing
    cache = {}

    # Prepare directories if saving any artifacts
    _dirs = init_results_dirs(event=event if save_results else None) if save_results else {}

    with tqdm(total=len(timestamps), desc="Processing times", unit="", colour="green") as pbar:
        start_time = time.time()
        for i, timestamp in enumerate(timestamps):
            # Format timestamp for all downstream use
            formatted_timestamp = format_timestamp(timestamp)
            subset = df[df['time'] == timestamp]
            event_name = subset['Event'].unique()[0]
            pbar.set_postfix_str(f"Current Timestamp: {formatted_timestamp}")

            # Clean outliers
            subset_cleaned = subset.pipe(clean_subset)

            # Features and labels
            subset_features1 = subset_cleaned[subset_cleaned.columns.intersection(features)]
            subset_features_renamed = subset_features1.rename(columns=custom_mapping)
            subset_features = subset_features_renamed.drop(columns=['Condition'])
            original_labels = list(subset_features.columns)

            # Colors
            colors = [
                '#71898E' if condition == 1 else '#cc2936' # autonomous
                for condition in subset_features_renamed['Condition']
            ]

            # Scaling
            subset_scaled = StandardScaler(with_std=True).fit_transform(subset_features) if scaled else subset_features
       
            # PCA
            # Set whiten to True to normalize the variance of the PCs 
            # No whitening: Cov(Y)=Λ (diagonal eigenvalues).
            # with whitening: Cov(Y)=I (identity matrix). (Cov of the transformed scores an identity matrix)
            pca = PCA(whiten=False, svd_solver='full') 
            pca_result = pca.fit_transform(subset_scaled)
            explained_variance = pca.explained_variance_ratio_
            eigenvalues = pca.explained_variance_

            # Eigenvalues DataFrame
            eigenvalues_df = pd.DataFrame(
                [eigenvalues],
                columns=[f'eigen_val{i+1}' for i in range(len(eigenvalues))]
            )
            eigenvalues_df = add_time_event(eigenvalues_df, formatted_timestamp, event_name)
            df_sorted_eigenvalues = pd.concat([df_sorted_eigenvalues, eigenvalues_df], ignore_index=True)

            # PCA results DataFrame
            pca_result_df = pd.DataFrame(
                pca_result,
                columns=[f'PC{i+1}' for i in range(len(explained_variance))]
            )
            pca_result_df = add_time_event(pca_result_df, formatted_timestamp, event_name)
            df_pca_results = pd.concat([df_pca_results, pca_result_df], ignore_index=True)

            # Components DataFrame
            loadings = pd.DataFrame(
                pca.components_.T,
                columns=[f'PC{i+1}' for i in range(len(explained_variance))]
            )
            loadings['Features'] = original_labels
            loadings = add_time_event(loadings, formatted_timestamp, event_name)
            df_event_components = pd.concat([df_event_components, loadings], ignore_index=True)

            # Variance DataFrame
            df_variances = pd.DataFrame(
                [explained_variance],
                columns=[f'PC{i+1}' for i in range(len(explained_variance))]
            )
            df_variances = add_time_event(df_variances, formatted_timestamp, event_name)
            df_event_variance = pd.concat([df_event_variance, df_variances], ignore_index=True)

            # Calculate eigenvalues Standard Deviation (as implemented in R prcomp())
            # each standard deviation is simply the square root of its corresponding eigenvalue.
            std_dev = np.sqrt(eigenvalues)
            df_std = pd.DataFrame(
                [std_dev],
                columns=[f'PC{i+1}_std' for i in range(len(explained_variance))]
            )
            df_std = add_time_event(df_std, formatted_timestamp, event_name)
            df_std_dev = pd.concat([df_std_dev, df_std], ignore_index=True)

            # Save all plots and cleaned data if requested
            if save_results:
                # Save biplot
                filename_biplot = f"biplot_event_{event or 'all_events'}_{formatted_timestamp}_{len(original_labels)}_vars{save_format}"
                plot_biplot(
                    pca_result, pca, eigenvalues, original_labels, colors, explained_variance,
                    show=False,
                    save_path=os.path.join(_dirs['biplots'], filename_biplot),
                    timestamp=formatted_timestamp
                )
                # Save Cos² plot
                filename_cos2 = f"cos2_event_{event or 'all_events'}_{formatted_timestamp}_{len(original_labels)}_vars{save_format}"
                plot_cos2(
                    pca, subset_scaled, original_labels,
                    show=False,
                    save_path=os.path.join(_dirs['cos_plots'], filename_cos2),
                    event=event,
                    timestamp=formatted_timestamp
                )
                # Save cleaned subset
                # Properly add uid and labels (Condition) to the cleaned DataFrame
                # Ensure that 'uid' and 'Condition' are present and aligned
                subset_features_renamed = add_time_event(subset_features_renamed, formatted_timestamp, event_name)
                # Add 'uid' and 'labels' columns from subset_cleaned, aligning by index
                subset_features_renamed['uid'] = subset_cleaned['uid'].values
                subset_features_renamed['labels'] = subset_cleaned['Condition'].values
                df_cleaned_ts = pd.concat([df_cleaned_ts, subset_features_renamed], ignore_index=True)

            # Do not show any plots if not interactive
            # Only store for interactive viewing
            cache[i] = {
                'timestamp': formatted_timestamp,
                'original_labels': original_labels,
                'colors': colors,
                'pca': pca,
                'pca_result': pca_result,
                'explained_variance': explained_variance,
                'eigenvalues': eigenvalues,
                'subset_scaled': subset_scaled,
            }

            # Update progress bar and elapsed time
            elapsed_time = time.time() - start_time
            pbar.update(1)
            pbar.set_postfix_str(f"Elapsed: {elapsed_time:.2f}s")

    # Save results dataframes as CSV if requested
    if save_results:
        if len(timestamps) > 1:
            timestamp_csv = 'all'
        else:
            # Use the formatted timestamp of the only time point
            timestamp_csv = format_timestamp(timestamps[0]) if len(timestamps) == 1 else 'all'
        data_dirs = init_results_dirs(event=event if event else None)
        n_vars = len(original_labels)
        def save_csv(df, subdir, prefix):
            df.to_csv(
                os.path.join(data_dirs[subdir], f"{prefix}_event_{event or 'all_events'}_timestamp_{str(timestamp_csv)}_{n_vars}_variables.csv"),
                index=False
            )
        save_csv(df_cleaned_ts, 'cleaned_new', 'cleaned')
        save_csv(df_event_variance, 'variances', 'variances')
        save_csv(df_event_components, 'components', 'components')
        save_csv(df_sorted_eigenvalues, 'eigenvalues', 'eigenval')
        # each std is simply the square root of its corresponding eigenvalue.
        # so they are both saved in the same folder
        save_csv(df_std_dev, 'eigenvalues', 'stddev') 

    # Interactive slider viewer (plots stacked) - viewing only, no saving here
    if interactive and len(timestamps) > 0:
        import ipywidgets as widgets
        from IPython.display import display

        slider = widgets.IntSlider(
            min=0,
            max=len(timestamps)-1,
            step=1,
            value=0,
            description='Time Point',
            layout=widgets.Layout(width='800px'),
            continuous_update=True,
        )

        def render(idx: int):
            comp = cache.get(idx)
            if comp is None:
                return
            plot_biplot(
                comp['pca_result'], comp['pca'], comp['eigenvalues'],
                comp['original_labels'], comp['colors'], comp['explained_variance'],
                show=True, save_path=None, timestamp=comp['timestamp']
            )
            plot_cos2(comp['pca'], comp['subset_scaled'], comp['original_labels'], show=True, save_path=None, timestamp=comp['timestamp'])

        out = widgets.interactive_output(render, {'idx': slider})
        display(slider, out)

    return df_event_components, df_event_variance, original_labels, df_cleaned_ts, df_pca_results, df_sorted_eigenvalues

## Variables for PCA

In [31]:
# Define the columns of the df to use in PCA
conditions = ['conditions']
euler_features = ['eye_theta_h', 'eye_theta_v',
                  'RelativeHeadYaw_degrees',
                  'RelativeHeadPitch_degrees',
                  'RelativeHeadRoll_degrees',
                  'CarYaw_degrees','streeringDegree']

**Define what we want to do:** We can use
- A subset of timestamps from the data `try_df`
- An specific event `resampled_df['Event'] == 'one'`
- Only visualize interactive or also save data
- Set `save_results = True` to save the data and plots in the respective `results` folder


For demonstration purposes, here we use some timestamps for event one. 

Change `try_df = resampled_df` to apply to the full drive

In [32]:
try_df = resampled_df[(resampled_df['Event'] == 'one') & (resampled_df['time'].isin([20.84, 23.74, 28.66,30.98]))]

In [ ]:
# choose event to work with or leave event = '' for all events 
event = ''
# choose features to use in PCA
features = conditions + euler_features
# To scale the data before PCA
scaled = True
# Interactive mode using slider to see PCA results at different timestamps
interactive=True
# Save results in the results folder
save_results=False

components_df, variances_df, original_labels, df_cleaned_ts,df_pca_results, eigenvalues_df = visualize_event_pca(try_df, event=event, features=features, scaled=scaled, save_results=save_results, interactive=interactive)

Processing times: 100%|██████████| 4/4 [00:00<00:00,  7.49/s, Elapsed: 0.53s]          


IntSlider(value=0, description='Time Point', layout=Layout(width='800px'), max=3)

Output()